<a href="https://colab.research.google.com/github/netsetos/genai-engg-gcp-learners/blob/main/module-01-setup-and-iam/lesson-1.3-first-gemini-call/practice/GCP_Capstone_1.3_Practice_Lab.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Practice Lab 1.3 — Your First Gemini Call

8 hands-on exercises with complete solutions. Master tokens, cost calculation, streaming, model comparison, and cost optimization.

Runnable companion to the published practice lab. Each exercise below shows the objective and a complete solution. Cloud Shell / `gcloud` steps are `%%bash` cells; Python steps run in Colab after you authenticate and set your project.

---

## Exercise 1: First Gemini Call & usage_metadata  
**Difficulty:** Easy

Make your first Gemini 3.6 Flash call. Print the response and ALL usage_metadata fields.

1. Initialize genai.Client with enterprise=True
2. Call generate_content with a simple prompt
3. Print response.text
4. Print all usage_metadata fields including thinking tokens

**Solution:**

In [ ]:
# --- Setup: install + auth (run me first) ---
!pip install -q google-genai
from google.colab import auth
auth.authenticate_user()

from google import genai
PROJECT_ID = 'documind-ai-YOUR-ID'   # CHANGE THIS to your project id
client = genai.Client(enterprise=True, project=PROJECT_ID, location='global')  # global: Gemini 3.x generation
print('Client ready')

In [ ]:
response = client.models.generate_content(
    model="gemini-3.6-flash",
    contents="What is a neural network? Answer in 2 sentences."
)

print("Response:", response.text)
print("\n📊 Token Breakdown:")
meta = response.usage_metadata
print(f"  Input tokens:    {meta.prompt_token_count}")
print(f"  Output tokens:   {meta.candidates_token_count}")
print(f"  Thinking tokens: {(getattr(meta, 'thoughts_token_count', 0) or 0)}")
print(f"  Total tokens:    {meta.total_token_count}")

## Exercise 2: Free Token Counter  
**Difficulty:** Easy

Use count_tokens() on 5 texts of increasing length. Find the tokens-per-character ratio.

1. Create 5 strings from 10 to 10,000 characters
2. Call count_tokens() on each (free, no cost)
3. Print tokens and chars/token ratio

**Solution:**

In [ ]:
texts = [
    "Hello world",
    "Hyderabad is the capital of Telangana in India",
    "The transformer architecture" * 20,
    "Explain retrieval augmented generation" * 100,
    "Large language models use self-attention" * 500,
]

print(f"  {'Chars':>8} {'Tokens':>8} {'Chars/Token':>12}")
for t in texts:
    r = client.models.count_tokens(model="gemini-3.6-flash", contents=t)
    ratio = len(t) / r.total_tokens
    print(f"  {len(t):>8} {r.total_tokens:>8} {ratio:>10.1f}:1")

## Exercise 3: INR Cost Calculator  
**Difficulty:** Easy

Build calc_cost_inr() and test on 3 different prompts. Show cost breakdown.

1. Define PRICING dict for 3 models
2. Build function taking usage_metadata + model name
3. Test on short, medium, and long prompts
4. Print input cost, output cost, thinking cost, total INR

**Solution:**

In [ ]:
PRICING = {
    "gemini-3.1-flash-lite": {"input": 0.25, "output": 1.50},
    "gemini-3.6-flash":      {"input": 1.50, "output": 7.50},
    "gemini-3.1-pro-preview":        {"input": 2.00, "output": 12.00},
}

def calc_cost_inr(meta, model="gemini-3.6-flash"):
    p = PRICING[model]
    inp = meta.prompt_token_count * p["input"] / 1_000_000
    out = (meta.candidates_token_count + (getattr(meta, "thoughts_token_count", 0) or 0)) * p["output"] / 1_000_000
    total = (inp + out) * 85
    print(f"  Input: ₹{inp*85:.4f} | Output: ₹{out*85:.4f} | Total: ₹{total:.4f}")
    return total

prompts = ["What is AI?", "Explain transformers in 5 sentences", "Design a complete RAG system" * 5]
for p in prompts:
    r = client.models.generate_content(model="gemini-3.6-flash", contents=p)
    print(f"\nPrompt: {p[:40]}...")
    calc_cost_inr(r.usage_metadata)

## Exercise 4: 3-Model Comparison Table  
**Difficulty:** Medium

Run the same prompt on Flash-Lite, Flash, and Pro. Build a comparison table with quality, latency, and cost.

1. Define prompt and 3 model configs
2. Time each call
3. Extract usage_metadata from each
4. Print formatted comparison table

**Solution:**

In [ ]:
import time

prompt = "Explain the difference between SQL and NoSQL databases. Give 2 examples of each."
models = [
    ("gemini-3.1-flash-lite", 0.25, 1.50),
    ("gemini-3.6-flash", 1.50, 7.50),
    ("gemini-3.1-pro-preview", 2.00, 12.00),
]

print(f"{'Model':<35} {'Tokens':>7} {'Latency':>9} {'₹ Cost':>9}")
print("-" * 65)
for name, ip, op in models:
    t0 = time.time()
    try:
        r = client.models.generate_content(model=name, contents=prompt)
        ms = (time.time()-t0)*1000
        u = r.usage_metadata
        cost = (u.prompt_token_count*ip + (u.candidates_token_count + (getattr(u, 'thoughts_token_count', 0) or 0))*op)/1e6*85
        print(f"{name:<35} {u.total_token_count:>7} {ms:>7.0f}ms ₹{cost:>7.4f}")
    except Exception as e:
        print(f"{name:<35} ERROR: {e}")

## Exercise 5: Thinking Budget Impact  
**Difficulty:** Medium

Same prompt with thinking_budget=0, 1024, and 8192. Compare token usage and cost.

1. Create 3 configs with different thinking budgets
2. Call same prompt with each config
3. Compare thinking tokens, total cost, and response quality

**Solution:**

In [ ]:
from google.genai import types

prompt = "What are the pros and cons of microservices architecture?"

budgets = [0, 1024, 8192]
print(f"  {'Budget':>8} {'Think':>7} {'Output':>8} {'Total':>7} {'₹ Cost':>9}")
for b in budgets:
    cfg = types.GenerateContentConfig(
        thinking_config=types.ThinkingConfig(thinking_budget=b)
    )
    r = client.models.generate_content(model="gemini-3.6-flash", contents=prompt, config=cfg)
    u = r.usage_metadata
    think = (getattr(u, "thoughts_token_count", 0) or 0)
    cost = (u.prompt_token_count*1.50 + (u.candidates_token_count+think)*7.50)/1e6*85
    print(f"  {b:>8} {think:>7} {u.candidates_token_count:>8} {u.total_token_count:>7} ₹{cost:>7.4f}")

## Exercise 6: Streaming Terminal Chatbot  
**Difficulty:** Medium

Build a multi-turn chatbot that streams Gemini responses token-by-token in the terminal.

1. Set up a while loop for user input
2. Call generate_content_stream for each message
3. Print chunks with end='' for streaming effect
4. Track total tokens and cost across the session

**Solution:**

In [ ]:
total_cost = 0
total_tokens = 0

print("🤖 DocuMind Chat (type 'quit' to exit)")
while True:
    user = input("\nYou: ")
    if user.lower() == "quit":
        break

    print("AI: ", end="")
    last_chunk = None
    for chunk in client.models.generate_content_stream(
        model="gemini-3.6-flash", contents=user
    ):
        print(chunk.text, end="", flush=True)
        last_chunk = chunk

    if last_chunk and last_chunk.usage_metadata:
        u = last_chunk.usage_metadata
        cost = (u.prompt_token_count*1.50 + (u.candidates_token_count + (getattr(u, 'thoughts_token_count', 0) or 0))*7.50)/1e6*85
        total_cost += cost
        total_tokens += u.total_token_count
        print(f"\n  [{u.total_token_count} tokens | ₹{cost:.4f} | Session: ₹{total_cost:.4f}]")

print(f"\n📊 Session total: {total_tokens} tokens | ₹{total_cost:.4f}")

## Exercise 7: Smart Model Router  
**Difficulty:** Challenge

Build a function that auto-classifies prompt complexity using Flash-Lite, then routes to the appropriate model.

1. Use Flash-Lite to classify prompt as simple/medium/complex
2. Route to Flash-Lite, Flash, or Pro based on classification
3. Track routing decisions and cost savings vs always-using-Pro

**Solution:**

In [ ]:
from google.genai import types

def smart_generate(prompt):
    # Step 1: Classify complexity with cheapest model
    classify = client.models.generate_content(
        model="gemini-3.1-flash-lite",
        contents=f"Classify this task as SIMPLE, MEDIUM, or COMPLEX. Reply with ONE word only: {prompt[:200]}",
        config=types.GenerateContentConfig(max_output_tokens=5, thinking_config=types.ThinkingConfig(thinking_budget=0))
    )
    level = (classify.text or '').strip().upper()
    if level not in {"SIMPLE","MEDIUM","COMPLEX"}: level = "MEDIUM"

    model_map = {"SIMPLE":"gemini-3.1-flash-lite", "MEDIUM":"gemini-3.6-flash", "COMPLEX":"gemini-3.1-pro-preview"}
    result = client.models.generate_content(model=model_map[level], contents=prompt)
    print(f"  Routed to: {level} → {model_map[level]}")
    return result

# Test routing
for p in ["Is Python interpreted?", "Explain attention mechanism", "Design a distributed RAG for 10M docs"]:
    print(f"\nPrompt: {p}")
    r = smart_generate(p)

## Exercise 8: Monthly Cost Projector  
**Difficulty:** Challenge

Build a tool that projects monthly Gemini API cost in INR given daily queries, token averages, and model mix.

1. Accept: queries/day, avg input/output tokens, model percentages
2. Calculate monthly cost for each model tier
3. Sum weighted costs
4. Project how many months $500 credits will last

**Solution:**

In [ ]:
def project_monthly(queries_day, avg_in, avg_out, mix={"flash-lite":0.6,"flash":0.3,"pro":0.1}):
    prices = {"flash-lite":(0.25,1.50), "flash":(1.50,7.50), "pro":(2.00,12.00)}
    monthly = queries_day * 30
    total = 0

    print(f"\n💰 Monthly Cost Projection")
    print(f"  Queries/day: {queries_day} | Monthly: {monthly:,}")
    print(f"  Avg tokens: {avg_in} in + {avg_out} out\n")

    for tier, pct in mix.items():
        ip, op = prices[tier]
        q = monthly * pct
        cost = (q * avg_in * ip + q * avg_out * op) / 1_000_000
        inr = cost * 85
        total += cost
        print(f"  {tier:<12} {pct*100:>4.0f}% | {q:>7,.0f} queries | ${cost:>6.2f} | ₹{inr:>8.2f}")

    total_inr = total * 85
    months = 500 / total if total > 0 else float("inf")
    print(f"\n  TOTAL: ${total:.2f}/month (₹{total_inr:.2f})")
    print(f"  $500 credits last: {months:.1f} months ({months/12:.1f} years)")

# DocuMind realistic usage
project_monthly(100, 2000, 500)